In [1]:
from typing import List, Dict
import os
from langchain.schema import Document
import json
from langchain_community.embeddings import OpenAIEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings

In [13]:
def load_corpus(paths: List[str]):
    """加载 BEIR corpus.jsonl"""
    corpus = []
    for path in paths:
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                doc = json.loads(line)
                corpus.append({
                    "id": doc["_id"],
                    "title": doc.get("title", ""),
                    "text": doc.get("text", "")
                })
    return corpus

In [14]:
corpus = load_corpus(["/workspace/zms/Data/rag-llm/data/fever/corpus.jsonl"])

In [43]:
MIN_WORDS = 500
long_docs = [
    doc
    for doc in corpus
    if len(doc["text"].split()) >= MIN_WORDS
]
print(f"✅ 从 {len(corpus)} 个文档中筛选出 {len(long_docs)} 个长文档（≥{MIN_WORDS} 词）")

✅ 从 5416568 个文档中筛选出 54633 个长文档（≥500 词）


In [68]:
def split_text_with_adaptive_overlap(text: str, chunk_size: int = 512, min_overlap: int = 64, max_overlap: int = 300, ratio: float = 0.15) -> list:
    """
    将 text 分块为字符串列表，overlap = clip(ratio * chunk_size, min_overlap, max_overlap)
    """
    words = text.split()
    if not words:
        return []

    overlap = max(min_overlap, min(max_overlap, int(chunk_size * ratio)))
    step = chunk_size - overlap
    # print(overlap, step)
    if step <= 0:
        step = chunk_size

    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunk_str = " ".join(words[start:end])
        if chunk_str.strip():
            chunks.append(chunk_str)
        if end >= len(words):
            break
        start += step

    return chunks, overlap


def split_text_without_overlap(text: str, chunk_size: int = 512) -> list:
    """
    将 text 分块为字符串列表，不考虑 overlap
    """
    words = text.split()
    if not words:
        return []

    overlap = 0
    step = chunk_size - overlap
    if step <= 0:
        step = chunk_size

    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunk_str = " ".join(words[start:end])
        if chunk_str.strip():
            chunks.append(chunk_str)
        if end >= len(words):
            break
        start += step

    return chunks, overlap

In [64]:
final_chunks = []
for doc in long_docs:
    # chunk_len = len(doc["text"].split())
    chunks, overlap = split_text_with_adaptive_overlap(
        text=doc["text"],
        chunk_size=768,
        min_overlap=64,
        max_overlap=300,
        ratio=0.3
    )
    # print(f"Document ID: {doc['id']} split into {len(chunks)} chunks with overlap {overlap}")
    count = 0
    for chunk_text in chunks:
        final_chunks.append({
            "id": f"{doc['id']}_{count}",
            "text": chunk_text,
            "title": doc["title"],
            "metadata": {"overlap": overlap}
        })
        count += 1

print(f"✅ 生成 {len(final_chunks)} 个 chunks")

✅ 生成 69993 个 chunks


In [65]:
final_chunks[0]

{'id': '1972–73_Chicago_Black_Hawks_season_0',
 'text': "The 1972 -- 73 Chicago Black Hawks season was the Hawks ' 47th season in the NHL , and the club was coming off their third consecutive first-place finish in 1971 -- 72 , as they finished on top of the West Division with a 46 -- 17 -- 15 record , tying a club record with 107 points . The Hawks swept the Pittsburgh Penguins in the NHL quarter-finals before they were swept by the New York Rangers in the semi-finals . During the off-season , the Black Hawks lost Bobby Hull to the Winnipeg Jets in the newly created World Hockey Association , who signed him to the first $ 1 million contract in hockey history . The NHL also expanded once again , as the New York Islanders would join the East Division , while the Atlanta Flames would be placed in the West . Chicago , dealing with the lost of Hull , got off to a quick start of the season , winning their first four games , before sliding into a slump that saw their record fall to 7 -- 7 -- 

In [67]:
output_file = "corpus.jsonl"
with open(output_file, 'w', encoding='utf-8') as f:
    for i, chunk in enumerate(final_chunks):
        new_chunk = {
            "id": chunk["id"],
            "title": chunk["title"],
            "text": chunk["text"],
            "metadata": chunk["metadata"]
        }
        f.write(json.dumps(new_chunk, ensure_ascii=False) + '\n')

In [69]:
final_chunks = []
for doc in long_docs:
    # chunk_len = len(doc["text"].split())
    chunks, overlap = split_text_without_overlap(
        text=doc["text"],
        chunk_size=768
    )
    # print(f"Document ID: {doc['id']} split into {len(chunks)} chunks with overlap {overlap}")
    count = 0
    for chunk_text in chunks:
        final_chunks.append({
            "id": f"{doc['id']}_{count}",
            "text": chunk_text,
            "title": doc["title"],
            "metadata": {"overlap": overlap}
        })
        count += 1

print(f"✅ 生成 {len(final_chunks)} 个 chunks")

✅ 生成 68110 个 chunks


In [70]:
output_file = "corpus.jsonl"
with open(output_file, 'w', encoding='utf-8') as f:
    for i, chunk in enumerate(final_chunks):
        new_chunk = {
            "id": chunk["id"],
            "title": chunk["title"],
            "text": chunk["text"],
            "metadata": chunk["metadata"]
        }
        f.write(json.dumps(new_chunk, ensure_ascii=False) + '\n')

In [71]:
def filter_queries_by_relevant_docs(queries, qrels, retained_doc_ids):
    """
    根据保留的文档 ID 集合，过滤 queries 和 qrels。
    
    Args:
        queries (dict): {"q1": "What is Du Fu known for?", ...}
        qrels (dict): {"q1": {"doc_id_1": 1, "doc_id_2": 0}, ...}
        retained_doc_ids (set): 保留的文档 ID 集合（如长文档）
    
    Returns:
        filtered_queries (dict), filtered_qrels (dict)
    """
    filtered_queries = {}
    filtered_qrels = {}

    for qid, query_text in queries.items():
        if qid not in qrels:
            continue
        
        # 获取该 query 的所有相关文档（label > 0）
        relevant_docs = {doc_id for doc_id, label in qrels[qid].items() if label > 0}
        
        # 检查是否有至少一个相关文档被保留
        if relevant_docs & retained_doc_ids:  # 集合交集非空
            filtered_queries[qid] = query_text
            # 同时过滤 qrels，只保留 retained_doc_ids 中的文档
            filtered_qrels[qid] = {
                doc_id: label
                for doc_id, label in qrels[qid].items()
                if doc_id in retained_doc_ids
            }

    return filtered_queries, filtered_qrels